In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('Low_yield_ligands_final_ligand.csv')




In [2]:
df.head()

,input,model_E_oxidation_boltz,model_E_reduction_boltz,model_E_solv_cds_boltz,model_E_solv_elstat_boltz,model_E_solv_total_boltz,model_Pint_P_int_boltz,model_Pint_P_max_boltz,model_Pint_P_min_boltz,model_Pint_dP_boltz,...,model_vbur_vbur_delta,model_vbur_vbur_max,model_vbur_vbur_min,model_vbur_vbur_vburminconf,model_vbur_vtot_boltz,model_vmin_r_boltz,model_vmin_vmin_boltz,model_volume_boltz,yield,SMILES
0,0000000000000000000000000000000001000000000001...,0.303620,0.076862,-2.404362,-3.899914,-5.572245,16.438795,26.658260,11.354961,3.364319,...,11.970101,55.491146,44.365550,44.634552,178.65941,1.829504,-0.053807,215.21457,0,CCP(CC)CC
1,0000000000000000000000000000000001000000000001...,0.315292,0.079952,0.100458,-5.185666,-3.445140,15.564138,25.196466,11.504475,3.012862,...,3.678765,42.227734,42.226490,43.004677,117.04840,1.835588,-0.048619,148.69606,0,CP(C)C
2,0000000000000000000000000000000000000000000000...,0.298519,0.066568,-1.781534,-4.142239,-6.172190,17.187168,27.688345,12.257298,3.036557,...,11.170786,57.019880,43.270283,40.860226,210.79329,1.892139,-0.042421,266.44974,0,[N@@]1(C[N@]2C3)CP3C[N@](C2)C1
3,0110100000000000000000000000000001000000000001...,0.282032,0.061337,-6.985048,-3.539417,-10.239269,18.203201,31.063185,11.856228,3.557869,...,24.009241,78.151146,53.715330,52.901450,297.37643,1.795672,-0.059711,341.08774,0,CC(CCC)P(C1CCCCC1)C2CCCCC2
4,0000000000000000000000000000000001001000000001...,0.267603,0.056862,-9.630296,-4.263012,-13.797570,19.652014,31.407488,11.884287,3.658416,...,21.787224,73.715370,58.651604,58.704190,395.04953,1.783544,-0.066182,493.20935,0,CCCCP([C@]1(C[C@H]2C3)C[C@@H](C2)C[C@@H]3C1)[C...


In [3]:
# Remove '.1' from all column names in the DataFrame
df.columns = df.columns.str.replace(r'\.1$', '', regex=True)


In [4]:
len(df)

37

In [5]:
df = df.drop(columns=['input', 'SMILES'])


In [7]:
total_non_zero_yields = (df['yield'] > 0).sum()
total_zero_yields = (df['yield'] == 0).sum()
total_non_zero_yields

22

In [8]:
import pandas as pd
import numpy as np

# Remove the first column (index 0) from the DataFrame
#df = df.drop(df.columns[0], axis=1)

# Ensure that the 'yield' column and all features are numeric
#df['yield'] = pd.to_numeric(df['yield'], errors='coerce')  # Convert 'yield' to numeric, invalid parsing will be set to NaN
#df = df.apply(pd.to_numeric, errors='coerce')  # Convert all features to numeric, invalid parsing will be set to NaN

# Drop rows with NaN values (if necessary, or handle them appropriately)
df = df.dropna()  # Or you could use df.fillna() to replace NaN with a value

best_thresholds = []

# Calculate the total non-zero and zero yields in the dataset
total_non_zero_yields = (df['yield'] > 0).sum()
total_zero_yields = (df['yield'] == 0).sum()

# Loop through each feature
for feature in df.columns:
    if feature != 'yield':  # Skip the target column
        min_val = df[feature].min()
        max_val = df[feature].max()
        
        # Define a range of potential thresholds (e.g., 100 intervals between min and max)
        thresholds = np.linspace(min_val, max_val, num=500)
        
        best_threshold = None
        max_separation = 0
        best_non_zero_before = 0
        best_zero_before = 0
        best_non_zero_after = 0
        best_zero_after = 0
        best_percent_non_zero_after = 0
        best_percent_zero_after = 0
        
        # Loop through all potential thresholds
        for threshold in thresholds:
            # Split the data into two parts: below and above the threshold
            before = df[df[feature] < threshold]
            after = df[df[feature] >= threshold]
            
            # Count non-zero and zero yields in both parts
            zero_before = (before['yield'] == 0).sum()
            non_zero_before = (before['yield'] > 0).sum()
            zero_after = (after['yield'] == 0).sum()
            non_zero_after = (after['yield'] > 0).sum()
            
            # Calculate the absolute difference in non-zero and zero yields on both sides
            separation_before = abs(non_zero_before - zero_before)
            separation_after = abs(non_zero_after - zero_after)
            
            # Calculate the percent of non-zero and zero yields after the threshold relative to total non-zero and zero yields
            percent_non_zero_after = (non_zero_after / total_non_zero_yields) * 100 if total_non_zero_yields > 0 else 0
            percent_zero_after = (zero_after / total_zero_yields) * 100 if total_zero_yields > 0 else 0
            
            # Total separation is the sum of separations on both sides
            total_separation = separation_before + separation_after
            
            # Update the best threshold if this separation is greater
            if total_separation > max_separation:
                max_separation = total_separation
                best_threshold = threshold
                best_non_zero_before = non_zero_before
                best_zero_before = zero_before
                best_non_zero_after = non_zero_after
                best_zero_after = zero_after
                best_percent_non_zero_after = percent_non_zero_after
                best_percent_zero_after = percent_zero_after
        
        # Store the results for this feature
        best_thresholds.append({
            'feature': feature,
            'best_threshold': best_threshold,
            'non_zero_before': best_non_zero_before,
            'zero_before': best_zero_before,
            'non_zero_after': best_non_zero_after,
            'zero_after': best_zero_after,
            'max_separation': max_separation,
            'percent_non_zero_after': best_percent_non_zero_after,
            'percent_zero_after': best_percent_zero_after
        })

# Convert results into a DataFrame for easier viewing
best_thresholds_df = pd.DataFrame(best_thresholds)
print(best_thresholds_df)


                         feature  best_threshold  non_zero_before  \
0        model_E_oxidation_boltz        0.265248               16   
1        model_E_reduction_boltz        0.023420               22   
2         model_E_solv_cds_boltz       -6.353836               22   
3      model_E_solv_elstat_boltz       -6.424860               22   
4       model_E_solv_total_boltz      -14.745712               18   
..                           ...             ...              ...   
185  model_vbur_vbur_vburminconf       50.045913                2   
186        model_vbur_vtot_boltz      413.578232                4   
187           model_vmin_r_boltz        1.863848               22   
188        model_vmin_vmin_boltz       -0.051700               19   
189           model_volume_boltz      453.701578                2   

     zero_before  non_zero_after  zero_after  max_separation  \
0              2               6          13              21   
1              8               0           

In [9]:
best_thresholds_df = best_thresholds_df.sort_values(by=['percent_non_zero_after','percent_zero_after'], ascending=[False, True])

best_thresholds_df.head()


,feature,best_threshold,non_zero_before,zero_before,non_zero_after,zero_after,max_separation,percent_non_zero_after,percent_zero_after
13,model_dipolemoment_vburminconf,1.493775,0,11,22,4,29,100.0,26.666667
7,model_Pint_P_min_boltz,12.489352,0,7,22,8,21,100.0,53.333333
59,model_qpoletens_xx_boltz,2.548305,0,7,22,8,21,100.0,53.333333
8,model_Pint_dP_boltz,3.701526,0,6,22,9,19,100.0,60.000000
14,model_efg_amp_P_boltz,1.741942,0,6,22,9,19,100.0,60.000000


In [10]:
best_thresholds_df.to_csv('Low_yield_ligands_final_sorted_best_thresholds.csv', index=False)


In [11]:
best_thresholds_df.head()

,feature,best_threshold,non_zero_before,zero_before,non_zero_after,zero_after,max_separation,percent_non_zero_after,percent_zero_after
13,model_dipolemoment_vburminconf,1.493775,0,11,22,4,29,100.0,26.666667
7,model_Pint_P_min_boltz,12.489352,0,7,22,8,21,100.0,53.333333
59,model_qpoletens_xx_boltz,2.548305,0,7,22,8,21,100.0,53.333333
8,model_Pint_dP_boltz,3.701526,0,6,22,9,19,100.0,60.000000
14,model_efg_amp_P_boltz,1.741942,0,6,22,9,19,100.0,60.000000
